# 🔍 CausalLens — Lalonde Job Training (NSW) Analysis

**Causal Question**: Does participating in a subsidised job-training programme
causally increase a worker's 1978 earnings?

**Dataset**: Lalonde (1986) NSW experiment — the gold-standard benchmark in causal inference.
The NSW was a genuine randomised controlled trial, so we know the *true* ATE.
We use it here to verify that our observational methods recover the experimental ground truth.

| Variable | Role | Description |
|----------|------|-------------|
| `treat`  | Treatment | 1 = job training, 0 = control |
| `re78`   | Outcome  | 1978 earnings (USD) |
| `age`, `educ`, `black`, `hisp`, `married`, `nodegree`, `re74`, `re75` | Confounders | Demographics & prior earnings |

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
print('Libraries loaded ✓')

## 1. Load & Explore Data

In [ ]:
from data.load_data import load_lalonde, dataset_summary, LALONDE_TREATMENT, LALONDE_OUTCOME, LALONDE_CONFOUNDERS

df = load_lalonde()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
summary = dataset_summary(df, LALONDE_TREATMENT, LALONDE_OUTCOME)
print('Dataset summary:')
for k, v in summary.items():
    print(f'  {k:30s}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Outcome distribution by treatment
for grp, label, color in [(1, 'Treated', '#2ecc71'), (0, 'Control', '#e74c3c')]:
    axes[0].hist(df.loc[df['treat']==grp, 're78'], bins=30,
                 alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('1978 Earnings by Treatment Status')
axes[0].set_xlabel('re78 (USD)')
axes[0].legend()

# Prior earnings comparison
df_melt = df[['treat','re74','re75']].melt(id_vars='treat')
sns.boxplot(data=df_melt, x='variable', y='value', hue='treat', ax=axes[1],
            palette={0: '#e74c3c', 1: '#2ecc71'})
axes[1].set_title('Prior Earnings (re74 / re75)')
axes[1].set_xlabel('')

# Balance check: education
df.groupby('treat')['educ'].plot(kind='kde', ax=axes[2])
axes[2].set_title('Education Distribution')
axes[2].set_xlabel('Years of Education')
axes[2].legend(['Control', 'Treated'])

plt.tight_layout()
plt.show()

## 2. Causal DAG

In [ ]:
from causal.discovery import build_causal_graph, plot_causal_graph

G = build_causal_graph('lalonde', LALONDE_TREATMENT, LALONDE_OUTCOME)
fig = plot_causal_graph(G, LALONDE_TREATMENT, LALONDE_OUTCOME,
                        title='Causal DAG — Lalonde NSW')
plt.show()
print(f'Nodes: {list(G.nodes())}')
print(f'Causal path: treat → re78')

## 3. Causal Effect Estimation

In [ ]:
from causal.effect_estimation import estimate_all_effects

results = estimate_all_effects(
    df,
    treatment=LALONDE_TREATMENT,
    outcome=LALONDE_OUTCOME,
    confounders=LALONDE_CONFOUNDERS,
    dataset_name='Lalonde NSW',
    graph=G,
)

print(f'\nNaive ATE         : {results.naive_ate:+.2f}')
print(f'PSM ATE           : {results.psm_ate:+.2f}')
print(f'IPW ATE           : {results.ipw_ate:+.2f}')
print(f'Double ML ATE     : {results.dml_ate:+.2f}  95% CI [{results.dml_ci_lower:.2f}, {results.dml_ate_upper:.2f}]')
print(f'Causal Forest ATE : {results.cfdml_ate:+.2f}  95% CI [{results.cfdml_ci_lower:.2f}, {results.cfdml_ci_upper:.2f}]')
print(f'\n(Experimental ground truth from Lalonde 1986: ~$886)')

In [ ]:
summary_df = results.summary_df()
display(summary_df)

fig, ax = plt.subplots(figsize=(8, 3.5))
ates = summary_df['ATE'].astype(float)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in ates]
bars = ax.barh(summary_df['Method'], ates, color=colors, height=0.5)
ax.axvline(0, color='#333', lw=1.2, ls='--')
ax.axvline(results.naive_ate, color='#f39c12', lw=1.5, ls=':', label=f'Naive ATE ({results.naive_ate:+.1f})')
ax.axvline(886, color='#8e44ad', lw=1.5, ls='-.', label='Experimental truth (~886)')
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=9)
ax.set_xlabel('ATE (USD)')
ax.set_title('Causal Effect Estimates vs Ground Truth', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Refutation Tests

In [ ]:
from causal.refutation import run_all_refutations, refutations_to_df

refutations = run_all_refutations(
    df, LALONDE_TREATMENT, LALONDE_OUTCOME, LALONDE_CONFOUNDERS,
    original_ate=results.dml_ate or 886.0,
    n_simulations=100,
)

display(refutations_to_df(refutations))

for r in refutations:
    icon = '✅' if r.passed else '⚠️'
    print(f'{icon} {r.test_name}: {r.interpretation}')

## 5. LLM Business Report (requires API key)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

from llm.report_generator import generate_full_report

dataset_desc = (
    'Lalonde NSW Job-Training dataset. We estimate whether participating '
    'in a job-training programme causally increases 1978 earnings, '
    'controlling for demographics and prior earnings.'
)

report = generate_full_report(results, refutations, dataset_desc)
print(report)